## 🛠️ Funciones auxiliares de visualización

Ejecuta la siguiente celda **una sola vez** al inicio. Después sólo tienes que **llamar** a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `plot_distributions(df, columnas)` | Histograma + boxplot de variables numéricas |
| `plot_frequencies(df, columnas, top_n=None)` | Frecuencia de variables categóricas |
| `plot_correlation_matrix(df, columnas)` | Matriz de correlación |
| `plot_pairplot(df, columnas, color=None)` | Dispersión entre todas las variables numéricas |
| `plot_simple_regression(x, y, results)` | Recta ajustada de un modelo OLS con 1 variable |
| `plot_actual_vs_predicted(y_real, y_pred)` | Valores reales vs predichos |
| `plot_residuals(y_real, y_pred)` | Residuales vs predichos |
| `plot_rfecv(rfecv)` | R² según el número de variables seleccionadas por RFECV |

In [1]:
# Funciones auxiliares de visualización
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


def plot_distributions(df, columns, nbins=30):
    """Histograma con boxplot marginal para cada variable numérica."""
    for col in columns:
        fig = px.histogram(
            df,
            x=col,
            nbins=nbins,
            marginal='box',
            opacity=0.7,
            title=f'Distribución de {col}'
        )
        fig.update_layout(bargap=0.2)
        fig.show()


def plot_frequencies(df, columns, top_n=None):
    """Gráfica de barras con la frecuencia de cada categoría (top_n limita a las más comunes)."""
    for col in columns:
        freq = df[col].value_counts()
        if top_n:
            freq = freq.head(top_n)
        freq_df = freq.rename_axis(col).reset_index(name='Frecuencia')

        title = f'Frecuencias de {col}'
        if top_n and df[col].nunique() > top_n:
            title += f' (top {top_n})'

        fig = px.bar(freq_df, x=col, y='Frecuencia', title=title)
        fig.update_layout(xaxis={'categoryorder': 'total descending'})
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def plot_pairplot(df, columns, color=None):
    """Matriz de dispersión (pairplot) entre las variables numéricas."""
    fig = px.scatter_matrix(
        df,
        dimensions=columns,
        color=color,
        title='Pairplot de Variables Numéricas',
        labels={col: col.capitalize() for col in columns}
    )
    fig.update_layout(width=1200, height=1200, title_font_size=20)
    fig.update_traces(diagonal_visible=True)
    fig.show()


def plot_simple_regression(x, y, results):
    """Dispersión de una variable vs el objetivo con la recta ajustada por un OLS de 1 variable."""
    b0, b1 = results.params.iloc[0], results.params.iloc[1]
    x_name = getattr(x, 'name', None) or 'x'
    y_name = getattr(y, 'name', None) or 'y'
    x_line = np.linspace(np.min(x), np.max(x), 100)

    fig = px.scatter(
        x=np.asarray(x),
        y=np.asarray(y),
        opacity=0.6,
        labels={'x': x_name, 'y': y_name},
        title=f'{y_name} = {b0:.2f} + ({b1:.4f}) · {x_name}',
        template='plotly_white'
    )
    fig.add_trace(go.Scatter(
        x=x_line,
        y=b0 + b1 * x_line,
        mode='lines',
        name='Recta OLS',
        line=dict(color='red', width=3)
    ))
    fig.show()


def plot_actual_vs_predicted(y_true, y_pred, title='Real vs Predicho'):
    """Valores reales vs predichos; un modelo perfecto cae sobre la diagonal roja."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())

    fig = px.scatter(
        x=y_true,
        y=y_pred,
        opacity=0.5,
        labels={'x': 'Valor real', 'y': 'Valor predicho'},
        title=title,
        template='plotly_white'
    )
    fig.add_shape(
        type='line', x0=lo, y0=lo, x1=hi, y1=hi,
        line=dict(color='red', dash='dash')
    )
    fig.show()


def plot_residuals(y_true, y_pred, title='Residuales vs Predicho'):
    """Residuales vs predichos; buscamos una nube sin patrón alrededor de 0."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)

    fig = px.scatter(
        x=y_pred,
        y=y_true - y_pred,
        opacity=0.5,
        labels={'x': 'Valor predicho', 'y': 'Residual (real − predicho)'},
        title=title,
        template='plotly_white'
    )
    fig.add_hline(y=0, line_dash='dash', line_color='red')
    fig.show()


def plot_rfecv(rfecv):
    """R² promedio de validación cruzada según el número de variables que conserva RFECV."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=rfecv.cv_results_['n_features'],
        y=rfecv.cv_results_['mean_test_score'],
        mode='lines+markers',
        line=dict(color='steelblue', width=3),
        marker=dict(size=7),
        name='R² promedio (CV)'
    ))
    fig.update_layout(
        title='RFECV — R² según número de variables seleccionadas',
        xaxis_title='Número de variables',
        yaxis_title='R² (validación cruzada)',
        template='plotly_white',
        width=900, height=450
    )
    fig.show()

In [2]:
import requests, sqlite3, pandas as pd

url = "https://raw.githubusercontent.com/davidjamesknight/SQLite_databases_for_learning_data_science/main/diamonds.db"
r = requests.get(url)

with open("diamonds.db", "wb") as f:
    f.write(r.content)

conn = sqlite3.connect("diamonds.db")

query = """
SELECT
  O.carat,
  O.price,
  O.depth,
  "O"."table",
  O.x,
  O.y,
  O.z,
  C.cut,
  Co.color,
  Cl.clarity
FROM
  Observation AS O
JOIN
  Cut AS C ON O.cut_id = C.cut_id
JOIN
  Color AS Co ON O.color_id = Co.color_id
JOIN
  Clarity AS Cl ON O.clarity_id = Cl.clarity_id
"""

df = pd.read_sql_query(query, conn)
df.head()

,carat,price,depth,table,x,y,z,cut,color,clarity
0,0.23,326,61.5,55.0,3.95,3.98,2.43,Ideal,E,SI2
1,0.21,326,59.8,61.0,3.89,3.84,2.31,Premium,E,SI1
2,0.23,327,56.9,65.0,4.05,4.07,2.31,Good,E,VS1
3,0.29,334,62.4,58.0,4.20,4.23,2.63,Premium,I,VS2
4,0.31,335,63.3,58.0,4.34,4.35,2.75,Good,J,SI2


## 📋 Recap del Análisis Exploratorio

En el notebook anterior exploramos el dataset de **53,940 diamantes**. Aquí un resumen rápido:

### Variables
| Tipo | Variables |
|---|---|
| **Numéricas** | `carat`, `depth`, `table`, `x`, `y`, `z`, `price` |
| **Categóricas (ordinales)** | `cut` (Fair → Ideal), `color` (D → J), `clarity` (IF → I1) |

### Hallazgos clave
- 💎 **`carat`**, **`x`**, **`y`** y **`z`** tienen la correlación más fuerte con `price`
- ⚠️ `depth` y `table` muestran alta dispersión y outliers elevados
- 📊 `price` y `carat` tienen distribuciones **sesgadas a la derecha**
- 🔗 Las variables de dimensión (`x`, `y`, `z`) están **altamente correlacionadas entre sí** → posible multicolinealidad

### Variable objetivo
> Predeciremos **`price`** (precio en USD) usando las demás características del diamante.

## 1️⃣ Preprocesamiento

Antes de ajustar el modelo, preparamos los datos:
- Separamos la variable objetivo (`price`) de las variables predictoras
- Dividimos en **80% entrenamiento / 20% prueba**

In [3]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (43152, 9) | Test: (10788, 9)


### Encoding de variables categóricas

Las variables `cut`, `color` y `clarity` son categóricas ordinales.  
Usamos **OrdinalEncoder** respetando el orden natural de cada una.

In [4]:
from sklearn.preprocessing import OrdinalEncoder

# Orden natural de cada variable categórica ordinal
cut_order      = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
color_order    = ['J', 'I', 'H', 'G', 'F', 'E', 'D']   # D es el mejor
clarity_order  = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

enc = OrdinalEncoder(
    categories=[
        cut_order, 
        color_order, 
        clarity_order
    ]
)

cat_cols = ['cut', 'color', 'clarity']

# Fit en train, transform en ambos
X_train_enc = X_train.copy()
X_test_enc  = X_test.copy()

X_train_enc[cat_cols] = enc.fit_transform(X_train[cat_cols])
X_test_enc[cat_cols]  = enc.transform(X_test[cat_cols])

X_train_enc.head()

,carat,depth,table,x,y,z,cut,color,clarity
26546,2.01,58.1,64.0,8.23,8.19,4.77,1.0,4.0,1.0
9159,1.01,60.0,60.0,6.57,6.49,3.92,2.0,5.0,1.0
14131,1.10,62.5,58.0,6.59,6.54,4.10,3.0,2.0,3.0
15757,1.50,61.5,65.0,7.21,7.17,4.42,1.0,5.0,1.0
24632,1.52,62.1,57.0,7.27,7.32,4.53,2.0,3.0,4.0


## 2️⃣ Ajuste del Modelo — Regresión Lineal (OLS)

Ajustamos un modelo **crudo** con todas las variables, sin ningún tipo de selección.  
Usamos `statsmodels` para ver el resumen estadístico completo.

In [5]:
import statsmodels.api as sm

X_train_const = sm.add_constant(X_train_enc)

model_sm = sm.OLS(y_train, X_train_const)
results  = model_sm.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.907
Model:                            OLS   Adj. R-squared:                  0.907
Method:                 Least Squares   F-statistic:                 4.694e+04
Date:                Mon, 21 Sep 2026   Prob (F-statistic):               0.00
Time:                        14:41:40   Log-Likelihood:            -3.6770e+05
No. Observations:               43152   AIC:                         7.354e+05
Df Residuals:                   43142   BIC:                         7.355e+05
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       3830.8354    472.330      8.110      0.0

## 3️⃣ Evaluación en Test

Medimos el desempeño del modelo con datos que **nunca vio durante el entrenamiento**.

In [6]:
from sklearn.metrics import r2_score, mean_squared_error
from math import sqrt

X_test_const = sm.add_constant(X_test_enc)
y_pred = results.predict(X_test_const)

r2   = r2_score(y_test, y_pred)
rmse = sqrt(mean_squared_error(y_test, y_pred))

print(f"R²   en test: {r2:.4f}")
print(f"RMSE en test: {rmse:.2f} USD")

R²   en test: 0.9057
RMSE en test: 1224.60 USD


In [7]:
# Real vs predicho y residuales del modelo baseline
plot_actual_vs_predicted(y_test, y_pred, title='OLS baseline — Real vs Predicho (test)')
plot_residuals(y_test, y_pred, title='OLS baseline — Residuales vs Predicho')

## 4️⃣ Experimento: Log-transformación de `price` y `carat`

Las variables `price` y `carat` tienen distribuciones **sesgadas a la derecha**.  
Aplicar `log` las acerca a una distribución normal, lo que puede mejorar el ajuste lineal.

> **¿Qué cambia?**  
> - Ahora el modelo predice **log(price)**, no price directamente  
> - El R² sí es comparable directamente entre modelos (mide proporción de varianza explicada)  
> - El RMSE y el MAPE **hay que revertirlos a USD** con `exp()` antes de comparar

> ⚠️ **Trampa común con el MAPE en escala log:**  
> Si calculas MAPE directamente sobre los valores en log, obtienes ~1.5% — parece excelente.  
> Pero estás dividiendo el error entre números como `log(5000) ≈ 8.5`, no entre `5000`.  
> Eso infla artificialmente la precisión. El MAPE real sobre precios en USD es ~11%.  
> **Siempre reporta MAPE en la escala original de la variable objetivo.**


In [8]:
# Distribución original de price y carat: ambas sesgadas a la derecha
plot_distributions(df, ['price', 'carat'])

In [9]:
import numpy as np

# --- Log-transformación ---
y_train_log = np.log(y_train)
y_test_log  = np.log(y_test)

# carat también tiene sesgo → log(carat)
X_train_log = X_train_enc.copy()
X_test_log  = X_test_enc.copy()

X_train_log['carat'] = np.log(X_train_enc['carat'])
X_test_log['carat']  = np.log(X_test_enc['carat'])

# --- Ajuste OLS con variables log-transformadas ---
X_train_log_const = sm.add_constant(X_train_log)
X_test_log_const  = sm.add_constant(X_test_log)

model_log = sm.OLS(y_train_log, X_train_log_const)
results_log = model_log.fit()

print(results_log.summary())


                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.979
Model:                            OLS   Adj. R-squared:                  0.979
Method:                 Least Squares   F-statistic:                 2.272e+05
Date:                Mon, 21 Sep 2026   Prob (F-statistic):               0.00
Time:                        14:41:40   Log-Likelihood:                 21860.
No. Observations:               43152   AIC:                        -4.370e+04
Df Residuals:                   43142   BIC:                        -4.361e+04
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          7.9246      0.070    112.509      0.0

In [10]:
from sklearn.metrics import mean_absolute_percentage_error

# --- Evaluación en test (escala log) ---
y_pred_log = results_log.predict(X_test_log_const)

r2_log   = r2_score(y_test_log, y_pred_log)

# --- Revertir a USD para comparar apples-to-apples ---
y_pred_usd = np.exp(y_pred_log)
rmse_usd   = sqrt(mean_squared_error(y_test, y_pred_usd))

# --- MAPE siempre en escala original (USD) ---
mape_baseline = mean_absolute_percentage_error(y_test, y_pred) * 100
mape_log      = mean_absolute_percentage_error(y_test, y_pred_usd) * 100

# --- Comparación ---
print("=" * 58)
print(f"{'Modelo':<25} {'R²':>6} {'RMSE (USD)':>12} {'MAPE':>8}")
print("=" * 58)
print(f"{'OLS baseline':<25} {r2:.4f}   {rmse:>10.2f}   {mape_baseline:>5.2f}%")
print(f"{'OLS log(price)+log(carat)':<25} {r2_log:.4f}   {rmse_usd:>10.2f}   {mape_log:>5.2f}%")
print("=" * 58)


Modelo                        R²   RMSE (USD)     MAPE
OLS baseline              0.9057      1224.60   44.57%
OLS log(price)+log(carat) 0.9789      1026.37   11.51%


In [11]:
# Real vs predicho y residuales del modelo log (en USD)
plot_actual_vs_predicted(y_test, y_pred_usd, title='OLS log(price)+log(carat) — Real vs Predicho (test, USD)')
plot_residuals(y_test, y_pred_usd, title='OLS log(price)+log(carat) — Residuales vs Predicho (USD)')